# Agente de Verificación de Listas Escolares - Librería Antártica

Este Jupyter Notebook integra el flujo multi-paso del agente:
1. Carga de datos de catálogo, stock y lista escolar.
2. Generación de embeddings con Gemini (`models/gemini-embedding-001`) y recuperación semántica (RAG).
3. Consulta de stock exacto por ISBN y sucursal.
4. Orquestación y consolidación de resultados.

In [1]:
# Instalación de las librerías necesarias segun requirements.txt
!pip install numpy google-generativeai python-dotenv
# Parte 2 (agente): framework LangChain + LangGraph con Groq, Mistral o Gemini
!pip install langchain langgraph langchain-google-genai langchain-mistralai langchain-groq fastembed langchain-core


[notice] A new release of pip is available: 26.1.2 -> 26.3
[notice] To update, run: python.exe -m pip install --upgrade pip



[notice] A new release of pip is available: 26.1.2 -> 26.3
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
import os
from dotenv import load_dotenv

# Cargar variables desde archivo .env si existe
load_dotenv()

# Configurar API Keys
os.environ["GOOGLE_API_KEY"] = os.getenv("GOOGLE_API_KEY", "TU_GOOGLE_API_KEY_AQUI")
os.environ["LLM_API_KEY"] = os.getenv("LLM_API_KEY", "TU_LLM_API_KEY_AQUI")

if os.environ["GOOGLE_API_KEY"] == "TU_GOOGLE_API_KEY_AQUI" and not (os.getenv("MISTRAL_API_KEY") or os.getenv("GROQ_API_KEY")):
    print("⚠️ Recuerda reemplazar 'TU_GOOGLE_API_KEY_AQUI' con tu API Key válida de Google AI Studio.")
# --- Proveedor de IA: "groq" (gratis, sin tarjeta), "mistral" o "gemini" ---
# Si no defines PROVEEDOR en el .env, se elige según qué clave exista (GROQ_API_KEY, MISTRAL_API_KEY, GOOGLE_API_KEY).
os.environ["MISTRAL_API_KEY"] = os.getenv("MISTRAL_API_KEY", "")
os.environ["GROQ_API_KEY"] = os.getenv("GROQ_API_KEY", "")
PROVEEDOR = (os.getenv("PROVEEDOR") or ("groq" if os.environ["GROQ_API_KEY"] else
             "mistral" if os.environ["MISTRAL_API_KEY"] else "gemini")).lower()
MODELO_MISTRAL = os.getenv("MISTRAL_MODEL", "mistral-small-latest")
MODELO_GROQ = os.getenv("GROQ_MODEL", "openai/gpt-oss-120b")
print("🔌 Proveedor de IA:", PROVEEDOR)

# En Google Colab: si no hay .env, intenta leer la clave desde Secrets (ícono de llave)
if os.environ["GOOGLE_API_KEY"] == "TU_GOOGLE_API_KEY_AQUI":
    try:
        from google.colab import userdata
        os.environ["GOOGLE_API_KEY"] = userdata.get("GOOGLE_API_KEY")
        print("🔑 API Key cargada desde Colab Secrets")
    except Exception:
        pass


🔌 Proveedor de IA: gemini


In [3]:
import os
import csv

# Crear carpeta 'data' si no existe
os.makedirs("data", exist_ok=True)

# 1. Muestra de Catálogo
catalogo_data = [
    {"isbn": "9789561234567", "titulo": "Papelucho", "autor": "Marcela Paz", "categoria": "Infantil", "edad_curso": "8 años / 3° Básico"},
    {"isbn": "9789561234568", "titulo": "El Principito", "autor": "Antoine de Saint-Exupéry", "categoria": "Literatura", "edad_curso": "10 años / 5° Básico"},
    {"isbn": "9789561234569", "titulo": "Sub Terra", "autor": "Baldomero Lillo", "categoria": "Narrativa Chilena", "edad_curso": "14 años / 1° Medio"}
]
# Ampliación de la muestra para la Parte 2 (los 3 títulos originales no cambian)
catalogo_data += [
    {"isbn": "9789561234570", "titulo": "Matilda", "autor": "Roald Dahl", "categoria": "Infantil", "edad_curso": "9 años / 4° Básico"},
    {"isbn": "9789561234571", "titulo": "Cuentos de la Selva", "autor": "Horacio Quiroga", "categoria": "Infantil", "edad_curso": "8 años / 3° Básico"},
    {"isbn": "9789561234572", "titulo": "Martín Rivas", "autor": "Alberto Blest Gana", "categoria": "Narrativa Chilena", "edad_curso": "15 años / 2° Medio"},
    {"isbn": "9789561234573", "titulo": "Hijo de Ladrón", "autor": "Manuel Rojas", "categoria": "Narrativa Chilena", "edad_curso": "16 años / 3° Medio"},
    {"isbn": "9789561234574", "titulo": "El Diario de Ana Frank", "autor": "Ana Frank", "categoria": "Biografía", "edad_curso": "13 años / 8° Básico"},
    {"isbn": "9789561234575", "titulo": "Mi Planta de Naranja Lima", "autor": "José Mauro de Vasconcelos", "categoria": "Literatura", "edad_curso": "12 años / 7° Básico"},
    {"isbn": "9789561234576", "titulo": "La Isla del Tesoro", "autor": "Robert Louis Stevenson", "categoria": "Aventura", "edad_curso": "11 años / 6° Básico"},
]
with open("data/catalogo_muestra.csv", "w", newline="", encoding="utf-8") as f:
    writer = csv.DictWriter(f, fieldnames=["isbn", "titulo", "autor", "categoria", "edad_curso"])
    writer.writeheader()
    writer.writerows(catalogo_data)

# 2. Muestra de Stock por Sucursal
stock_data = [
    {"isbn": "9789561234567", "sucursal": "Mall Costanera Center", "cantidad": "5"},
    {"isbn": "9789561234567", "sucursal": "Plaza Vespucio", "cantidad": "0"},
    {"isbn": "9789561234568", "sucursal": "Mall Costanera Center", "cantidad": "2"},
    {"isbn": "9789561234569", "sucursal": "Plaza Vespucio", "cantidad": "0"}
]
stock_data += [
    {"isbn": "9789561234570", "sucursal": "Plaza Vespucio", "cantidad": "6"},
    {"isbn": "9789561234570", "sucursal": "Mall Costanera Center", "cantidad": "3"},
    {"isbn": "9789561234571", "sucursal": "Mall Costanera Center", "cantidad": "4"},
    {"isbn": "9789561234572", "sucursal": "Mall Costanera Center", "cantidad": "3"},
    {"isbn": "9789561234572", "sucursal": "Plaza Vespucio", "cantidad": "1"},
    {"isbn": "9789561234573", "sucursal": "Plaza Vespucio", "cantidad": "2"},
    {"isbn": "9789561234574", "sucursal": "Mall Costanera Center", "cantidad": "0"},
    {"isbn": "9789561234574", "sucursal": "Plaza Vespucio", "cantidad": "0"},
    {"isbn": "9789561234575", "sucursal": "Mall Costanera Center", "cantidad": "7"},
    {"isbn": "9789561234576", "sucursal": "Plaza Vespucio", "cantidad": "5"},
]
with open("data/stock_sucursales.csv", "w", newline="", encoding="utf-8") as f:
    writer = csv.DictWriter(f, fieldnames=["isbn", "sucursal", "cantidad"])
    writer.writeheader()
    writer.writerows(stock_data)

# 3. Lista Escolar de Ejemplo
lista_data = [
    {"titulo": "Papelucho", "curso": "3° Básico"},
    {"titulo": "El Principito", "curso": "5° Básico"},
    {"titulo": "Sub Terra", "curso": "1° Medio"}
]
with open("data/lista_escolar_ejemplo.csv", "w", newline="", encoding="utf-8") as f:
    writer = csv.DictWriter(f, fieldnames=["titulo", "curso"])
    writer.writeheader()
    writer.writerows(lista_data)

# 4. Lista escolar ampliada (Parte 2): incluye un título sin stock ni alternativas y uno que no existe
lista_compleja = lista_data + [
    {"titulo": "Matilda", "curso": "4° Básico"},
    {"titulo": "El Diario de Ana Frank", "curso": "8° Básico"},
    {"titulo": "Mi Planta Naranja Lima", "curso": "7° Básico"},
    {"titulo": "Harry Potter y la piedra filosofal", "curso": "6° Básico"},
]
with open("data/lista_escolar_compleja.csv", "w", newline="", encoding="utf-8") as f:
    writer = csv.DictWriter(f, fieldnames=["titulo", "curso"])
    writer.writeheader()
    writer.writerows(lista_compleja)

print("✅ Archivos CSV de prueba creados en el directorio 'data/'")

✅ Archivos CSV de prueba creados en el directorio 'data/'


In [4]:
# Plantillas de prompt del agente de atención de Librería Antártica

SYSTEM_PROMPT = """Eres el asistente de atención al cliente de Librería Antártica.
Respondes solo en base al CONTEXTO entregado (catálogo, stock y listas escolares).
Si el dato no está en el contexto, indica que no tienes esa información y ofrece
derivar a un ejecutivo; nunca inventes disponibilidad, precio o título.
Cita siempre la sucursal y la fuente del dato. Responde en español, de forma
breve y cordial."""

PROMPT_DISPONIBILIDAD = """Consulta del cliente: {consulta_usuario}
CONTEXTO recuperado (top-k del catálogo y stock):
{contexto_rag}

Instrucciones: identifica el/los título(s) mencionados, confirma disponibilidad
por sucursal usando solo el CONTEXTO, y si no hay stock sugiere hasta 2
alternativas del mismo género/nivel presentes en el CONTEXTO."""

PROMPT_VERIFICACION_TITULO = """Título requerido: {titulo_lista}, curso: {curso}
CONTEXTO: {resultado_busqueda_catalogo_y_stock}

Devuelve solo un JSON con este formato exacto:
{{"titulo": "...", "disponible": true/false, "sucursal": "...", "cantidad": 0,
"alternativa_si_no_disponible": "..."}}"""

PROMPT_CONSOLIDACION = """A partir de esta lista de resultados por título:
{lista_resultados_json}

Redacta un resumen para el apoderado: cuántos títulos están disponibles, en qué
sucursal retirarlos, y cuáles requieren reemplazo o encargo, en un máximo de
8 líneas."""

In [5]:
from collections import defaultdict

def cargar_stock(path="data/stock_sucursales.csv"):
    stock = defaultdict(list)
    with open(path, newline="", encoding="utf-8") as f:
        for fila in csv.DictReader(f):
            stock[fila["isbn"]].append(
                {"sucursal": fila["sucursal"], "cantidad": int(fila["cantidad"])}
            )
    return stock

def disponibilidad_por_isbn(stock, isbn):
    """Devuelve las sucursales con stock > 0 para un ISBN dado."""
    registros = stock.get(isbn, [])
    return [r for r in registros if r["cantidad"] > 0]

In [6]:
import numpy as np

try:
    import google.generativeai as genai
except ImportError:
    genai = None

EMBED_MODEL = "models/gemini-embedding-001"

def cargar_catalogo(path="data/catalogo_muestra.csv"):
    with open(path, newline="", encoding="utf-8") as f:
        return list(csv.DictReader(f))

def construir_chunk(fila):
    """Une los campos relevantes de un producto en un solo texto (chunk)."""
    return (
        f"Título: {fila['titulo']}. Autor: {fila['autor']}. "
        f"Categoría: {fila['categoria']}. Edad/curso sugerido: {fila['edad_curso']}. "
        f"ISBN: {fila['isbn']}."
    )

def embed_textos_gemini(textos):
    if genai is None:
        raise RuntimeError("Instala google-generativeai (ver requirements.txt)")
    genai.configure(api_key=os.environ["GOOGLE_API_KEY"])
    vectores = []
    for t in textos:
        resp = genai.embed_content(model=EMBED_MODEL, content=t)
        vectores.append(resp["embedding"])
    return np.array(vectores)

def construir_indice(catalogo):
    chunks = [construir_chunk(f) for f in catalogo]
    vectores = embed_textos(chunks)
    return {"catalogo": catalogo, "chunks": chunks, "vectores": vectores}

def buscar(indice, consulta, top_k=3):
    """Recupera los top_k productos más similares a la consulta."""
    vec_consulta = embed_textos([consulta])[0]
    sims = indice["vectores"] @ vec_consulta / (
        np.linalg.norm(indice["vectores"], axis=1) * np.linalg.norm(vec_consulta) + 1e-8
    )
    top_idx = np.argsort(-sims)[:top_k]
    return [indice["catalogo"][i] for i in top_idx]

def embed_textos_mistral(textos):
    """Embeddings con Mistral (mistral-embed), en un solo lote."""
    global _EMB_MISTRAL
    if "_EMB_MISTRAL" not in globals():
        from langchain_mistralai import MistralAIEmbeddings
        _EMB_MISTRAL = MistralAIEmbeddings(model="mistral-embed", api_key=os.environ["MISTRAL_API_KEY"], max_retries=6)
    return np.array(_EMB_MISTRAL.embed_documents(list(textos)))

def _embed_hash(textos, dim=512):
    """Respaldo sin dependencias: embeddings por n-gramas de caracteres (captura parecido textual)."""
    import hashlib, unicodedata
    out = []
    for t in textos:
        t = unicodedata.normalize("NFD", t.lower())
        t = " " + "".join(c for c in t if unicodedata.category(c) != "Mn") + " "
        v = np.zeros(dim)
        for n in (3, 4):
            for i in range(len(t) - n + 1):
                v[int(hashlib.md5(t[i:i + n].encode()).hexdigest(), 16) % dim] += 1
        out.append(v / (np.linalg.norm(v) + 1e-9))
    return np.array(out)

def embed_textos_local(textos):
    """Embeddings LOCALES y gratis (sin API key). Usa fastembed con un modelo multilingüe;
    si no está disponible, cae a n-gramas de caracteres."""
    global _EMB_LOCAL
    if "_EMB_LOCAL" not in globals():
        try:
            from fastembed import TextEmbedding
            _EMB_LOCAL = TextEmbedding(model_name="sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2")
            print("🧩 Embeddings locales: fastembed (paraphrase-multilingual-MiniLM-L12-v2)")
        except Exception as e:
            _EMB_LOCAL = None
            print(f"ℹ️ fastembed no disponible ({str(e)[:80]}). Usando embeddings por n-gramas de caracteres.")
    if _EMB_LOCAL is None:
        return _embed_hash(list(textos))
    return np.array(list(_EMB_LOCAL.embed(list(textos))))

def embed_textos(textos):
    """Punto único de embeddings según el proveedor. Groq no ofrece embeddings: usa los locales."""
    if PROVEEDOR == "mistral":
        return embed_textos_mistral(textos)
    if PROVEEDOR == "gemini":
        return embed_textos_gemini(textos)
    return embed_textos_local(textos)


c:\Users\tatim\AppData\Local\Programs\Python\Python314\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
C:\Users\tatim\AppData\Local\Temp\ipykernel_16508\657928395.py:4: FutureWarning: 

All support for the `google.generativeai` package has ended. It will no longer be receiving 
updates or bug fixes. Please switch to the `google.genai` package as soon as possible.
See README for more details:

https://github.com/google-gemini/deprecated-generative-ai-python/blob/main/README.md

  import google.generativeai as genai


In [7]:
import json

def cargar_lista_escolar(path="data/lista_escolar_ejemplo.csv"):
    with open(path, newline="", encoding="utf-8") as f:
        return list(csv.DictReader(f))

def verificar_lista(path_lista="data/lista_escolar_ejemplo.csv"):
    catalogo = cargar_catalogo()
    indice = construir_indice(catalogo)
    stock = cargar_stock()
    lista = cargar_lista_escolar(path_lista)

    resultados = []
    for item in lista:
        candidatos = buscar(indice, item["titulo"], top_k=1)
        if not candidatos:
            resultados.append({"titulo": item["titulo"], "disponible": False,
                                "sucursal": None, "cantidad": 0,
                                "alternativa_si_no_disponible": None})
            continue
        producto = candidatos[0]
        disponibles = disponibilidad_por_isbn(stock, producto["isbn"])
        if disponibles:
            mejor = max(disponibles, key=lambda r: r["cantidad"])
            resultados.append({"titulo": producto["titulo"], "disponible": True,
                                "sucursal": mejor["sucursal"], "cantidad": mejor["cantidad"],
                                "alternativa_si_no_disponible": None})
        else:
            resultados.append({"titulo": producto["titulo"], "disponible": False,
                                "sucursal": None, "cantidad": 0,
                                "alternativa_si_no_disponible": "Consultar encargo"})

    return resultados

# Ejecución del agente
resultados = verificar_lista()
print("Resultados consolidados de la verificación:")
print(json.dumps(resultados, indent=2, ensure_ascii=False))

Resultados consolidados de la verificación:
[
  {
    "titulo": "Papelucho",
    "disponible": true,
    "sucursal": "Mall Costanera Center",
    "cantidad": 5,
    "alternativa_si_no_disponible": null
  },
  {
    "titulo": "El Principito",
    "disponible": true,
    "sucursal": "Mall Costanera Center",
    "cantidad": 2,
    "alternativa_si_no_disponible": null
  },
  {
    "titulo": "Sub Terra",
    "disponible": false,
    "sucursal": null,
    "cantidad": 0,
    "alternativa_si_no_disponible": "Consultar encargo"
  }
]


In [8]:
# Ejemplo de invocación del LLM para consolidar el resumen final usando Gemini
# gemini-1.5-flash fue retirado por Google: el modelo se configura por variable de entorno
MODELO_LLM_RESUMEN = os.getenv("GEMINI_MODEL", "gemini-flash-latest")

def consolidar_resumen_llm(resultados):
    if PROVEEDOR == "groq":
        from langchain_groq import ChatGroq
        llm = ChatGroq(model=MODELO_GROQ, api_key=os.environ["GROQ_API_KEY"], temperature=0)
        prompt = PROMPT_CONSOLIDACION.format(lista_resultados_json=json.dumps(resultados, ensure_ascii=False))
        return llm.invoke([("system", SYSTEM_PROMPT), ("human", prompt)]).content
    if PROVEEDOR == "mistral":
        from langchain_mistralai import ChatMistralAI
        llm = ChatMistralAI(model=MODELO_MISTRAL, api_key=os.environ["MISTRAL_API_KEY"], temperature=0)
        prompt = PROMPT_CONSOLIDACION.format(lista_resultados_json=json.dumps(resultados, ensure_ascii=False))
        return llm.invoke([("system", SYSTEM_PROMPT), ("human", prompt)]).content
    genai.configure(api_key=os.environ["GOOGLE_API_KEY"])
    model = genai.GenerativeModel(MODELO_LLM_RESUMEN, system_instruction=SYSTEM_PROMPT)
    
    prompt = PROMPT_CONSOLIDACION.format(
        lista_resultados_json=json.dumps(resultados, ensure_ascii=False)
    )
    
    response = model.generate_content(prompt)
    return response.text

# Generar resumen consolidado
try:
    resumen = consolidar_resumen_llm(resultados)
    print("--- Resumen para el Apoderado ---")
    print(resumen)
except Exception as e:
    print(f"Error al generar el resumen con el LLM: {e}")

Error al generar el resumen con el LLM: 429 You exceeded your current quota, please check your plan and billing details. For more information on this error, head to: https://aistudio.google.com/docs/rate-limits. To monitor your current usage, head to: https://ai.dev/rate-limit. 
* Quota exceeded for metric: generativelanguage.googleapis.com/generate_content_free_tier_requests, limit: 20, model: gemini-3.8-flash, description: Generate content requests per day limit exceeded on Free Tier. See https://aistudio.google.com/docs/rate-limits.
Please retry in 6h55m38.741194852s. [links {
  description: "Learn more about Gemini API quotas"
  url: "https://aistudio.google.com/docs/rate-limits"
}
, violations {
  description: "Generate content requests per day limit exceeded on Free Tier. See https://aistudio.google.com/docs/rate-limits."
  quota_metric: "generativelanguage.googleapis.com/generate_content_free_tier_requests"
  quota_id: "GenerateRequestsPerDayPerProjectPerModel-FreeTier"
  quota_

---
# 🤖 PARTE 2 — Agente funcional (Evaluación Parcial N°2)

Hasta aquí el notebook implementa el **pipeline RAG** de la Evaluación 1 (`verificar_lista()`), que es un flujo fijo. A partir de esta sección se agrega un **agente** construido con **LangChain + LangGraph** (`create_agent`, patrón ReAct) que decide por sí mismo qué herramienta usar en cada paso:

| Requisito de la pauta | Dónde se implementa |
|---|---|
| **IE1 – Herramientas** de consulta, escritura y razonamiento | Sección 2.3 (`@tool`) |
| **IE2 – Framework** de agentes | LangChain `create_agent` sobre LangGraph (sección 2.4) |
| **IE3 – Memoria** de corto y largo plazo | `InMemorySaver` + `EstadoSesion` + `MemoriaLargoPlazo` (secciones 2.2 y 2.4) |
| **IE4 – Recuperación semántica de contexto** | Embeddings Gemini sobre catálogo (`buscar_con_score`) y sobre la memoria (`MemoriaLargoPlazo.recuperar`) |
| **IE5 – Planificación y decisiones adaptativas** | `planificar_tareas()`, `decidir_retiro()` y reglas del `AGENT_SYSTEM_PROMPT` |
| **Evidencia de pruebas** | Secciones 2.5 y 2.6 (se guardan en `evidencias/`) |

> **Proveedor de IA:** el notebook funciona con **Groq** (gratis, sin tarjeta), **Mistral** o **Gemini**. Se elige con `PROVEEDOR` en el `.env`. Groq solo ofrece LLM, así que los embeddings se calculan **localmente y sin costo** (fastembed, modelo multilingüe).

> **Importante:** todo el código de la Evaluación 1 se mantiene sin cambios de lógica. Solo se amplió la muestra de datos (los 3 títulos originales siguen igual) y se actualizó el modelo del resumen, porque `gemini-1.5-flash` ya fue retirado por Google.


In [9]:
# ===== 2.1 Configuración y utilidades del agente =====
import re
import time
import warnings
warnings.filterwarnings("ignore", category=UserWarning)   # evita avisos repetidos de las librerías
import json
import difflib
import datetime
import unicodedata
from pathlib import Path

try:
    from langchain_core.tools import tool
    from langchain.agents import create_agent
    from langgraph.checkpoint.memory import InMemorySaver
    if PROVEEDOR == "groq":
        from langchain_groq import ChatGroq
    elif PROVEEDOR == "mistral":
        from langchain_mistralai import ChatMistralAI
    else:
        from langchain_google_genai import ChatGoogleGenerativeAI
except ImportError as e:
    raise ImportError(
        "Faltan dependencias del agente. Ejecuta la celda de instalación "
        "(langchain, langgraph, langchain-google-genai, langchain-mistralai, langchain-groq) y reinicia el kernel."
    ) from e

# --- Parámetros de decisión (se pueden ajustar sin tocar el resto del código) ---
# Los embeddings de cada proveedor tienen otra escala: ajusta estos umbrales viendo los puntajes impresos en la sección 2.5
UMBRAL_COSENO = float(os.getenv("UMBRAL_COSENO", "0.90" if PROVEEDOR == "mistral" else "0.80"))   # similitud semántica mínima para aceptar un título
UMBRAL_LEXICO = float(os.getenv("UMBRAL_LEXICO", "0.60"))   # similitud textual mínima (tolera errores de escritura)
UMBRAL_MEMORIA = float(os.getenv("UMBRAL_MEMORIA", "0.60" if PROVEEDOR == "mistral" else "0.35"))  # similitud mínima para recordar
STOCK_CRITICO = 2                                           # <= 2 unidades en la mejor sucursal => retirar pronto

# El modelo se prueba en este orden; el primero que responda se usa.
# Puedes forzar uno con GEMINI_MODEL en el .env
MODELOS_LLM = [m for m in [os.getenv("GEMINI_MODEL"), "gemini-3.8-flash",
                           "gemini-3.5-flash-lite", "gemini-flash-latest"] if m]

# Contexto compartido entre las herramientas (catálogo, índice, stock, memoria, LLM)
CTX = {"catalogo": None, "indice": None, "stock": None, "memoria": None,
       "llm": None, "thread_id": "default", "apoderado": "apoderado_demo"}

# --- Utilidades de texto ---
STOPWORDS = {"el", "la", "los", "las", "de", "del", "y", "un", "una", "en"}

def _tokens(texto):
    t = unicodedata.normalize("NFD", str(texto).lower())
    t = "".join(c for c in t if unicodedata.category(c) != "Mn")
    return re.sub(r"[^a-z0-9 ]", " ", t).split()

def _norm(texto):
    return " ".join(_tokens(texto))

def _texto(contenido):
    """El contenido de un mensaje de Gemini puede ser str o lista de bloques."""
    if isinstance(contenido, str):
        return contenido
    partes = []
    for bloque in contenido or []:
        if isinstance(bloque, str):
            partes.append(bloque)
        elif isinstance(bloque, dict) and bloque.get("type") == "text":
            partes.append(bloque.get("text", ""))
    return "".join(partes)

def _json(obj):
    return json.dumps(obj, ensure_ascii=False)

def _ahora():
    return datetime.datetime.now().strftime("%Y-%m-%d %H:%M:%S")

def similitud_lexica(consulta, titulo):
    """Similitud textual 0-1 tolerante a errores de tipeo y palabras extra ('papelucho 3 basico')."""
    tq = [w for w in _tokens(consulta) if w not in STOPWORDS]
    tt = [w for w in _tokens(titulo) if w not in STOPWORDS]
    if not tq or not tt:
        return 0.0
    contenidas = sum(
        1 for w in tt
        if max(difflib.SequenceMatcher(None, w, q).ratio() for q in tq) >= 0.8
    )
    contencion = contenidas / len(tt)
    razon = difflib.SequenceMatcher(None, " ".join(tq), " ".join(tt)).ratio()
    return max(contencion, razon)

def buscar_con_score(indice, consulta, top_k=3):
    """Versión de buscar() que además informa los puntajes y si el título realmente coincide.
    Ranking: coseno semántico (Gemini) con bonus si el título coincide textualmente.
    `coincide` evita aceptar siempre el 'menos malo' cuando el título no existe."""
    vq = embed_textos([consulta])[0]
    V = indice["vectores"]
    cos = V @ vq / (np.linalg.norm(V, axis=1) * np.linalg.norm(vq) + 1e-8)
    filas = []
    for i, producto in enumerate(indice["catalogo"]):
        lex = similitud_lexica(consulta, producto["titulo"])
        coincide = bool(cos[i] >= UMBRAL_COSENO or lex >= UMBRAL_LEXICO)
        filas.append({"producto": producto, "coseno": round(float(cos[i]), 3),
                      "lexico": round(float(lex), 3), "coincide": coincide,
                      "_orden": float(cos[i]) + (1.0 if lex >= UMBRAL_LEXICO else 0.0)})
    filas.sort(key=lambda r: -r["_orden"])
    for f in filas:
        f.pop("_orden")
    return filas[:top_k]

def obtener_llm():
    """Devuelve el primer modelo del proveedor elegido que responda (los modelos cambian con el tiempo)."""
    errores = []
    if PROVEEDOR == "groq":
        # Los modelos gratuitos de Groq cambian con frecuencia: se prueban varios en orden.
        for nombre in dict.fromkeys([MODELO_GROQ, "openai/gpt-oss-120b", "llama-3.3-70b-versatile",
                                     "openai/gpt-oss-20b"]):
            try:
                llm = ChatGroq(model=nombre, api_key=os.environ["GROQ_API_KEY"], temperature=0, max_retries=6)
                llm.invoke("Responde solo: ok")
                print(f"✅ Modelo LLM en uso (Groq): {nombre}")
                return llm
            except Exception as e:
                errores.append(f"{nombre}: {str(e)[:150]}")
    elif PROVEEDOR == "mistral":
        for nombre in dict.fromkeys([MODELO_MISTRAL, "mistral-small-latest", "mistral-medium-latest"]):
            try:
                llm = ChatMistralAI(model=nombre, api_key=os.environ["MISTRAL_API_KEY"],
                                    temperature=0, max_retries=6)
                llm.invoke("Responde solo: ok")
                print(f"✅ Modelo LLM en uso (Mistral): {nombre}")
                return llm
            except Exception as e:
                errores.append(f"{nombre}: {str(e)[:150]}")
    else:
        for nombre in MODELOS_LLM:
            try:
                llm = ChatGoogleGenerativeAI(model=nombre, google_api_key=os.environ["GOOGLE_API_KEY"],
                                             temperature=0)
                llm.invoke("Responde solo: ok")
                print(f"✅ Modelo LLM en uso (Gemini): {nombre}")
                return llm
            except Exception as e:  # modelo retirado, sin cuota, key inválida, etc.
                errores.append(f"{nombre}: {str(e)[:150]}")
    raise RuntimeError("Ningún modelo respondió:\n" + "\n".join(errores))

# --- Caché de embeddings: el mismo texto no se vuelve a enviar a la API (ahorra tiempo y cuota) ---
if "_embed_sin_cache" not in globals():
    _embed_sin_cache = embed_textos
    _CACHE_EMB = {}

    def embed_textos(textos):
        textos = list(textos)
        faltan = [t for t in dict.fromkeys(textos) if t not in _CACHE_EMB]
        if faltan:
            for t, v in zip(faltan, _embed_sin_cache(faltan)):
                _CACHE_EMB[t] = v
        return np.array([_CACHE_EMB[t] for t in textos])

print("✅ Configuración del agente cargada")


✅ Configuración del agente cargada


In [10]:
# ===== 2.2 Memoria del agente =====
# Memoria de CORTO plazo:
#   a) historial de la conversación por hilo (thread_id) -> checkpointer de LangGraph (sección 2.4)
#   b) EstadoSesion: "pizarra" con el plan y los resultados ya obtenidos en esta sesión.
# Memoria de LARGO plazo:
#   MemoriaLargoPlazo: hechos/preferencias del apoderado guardados en disco (JSON) con su embedding,
#   recuperables por similitud semántica en sesiones futuras.

ESTADOS = {}   # thread_id -> estado de la sesión

def estado_sesion(thread_id=None):
    tid = thread_id or CTX["thread_id"]
    return ESTADOS.setdefault(tid, {"plan": [], "resultados": {}, "iniciado": _ahora()})

class MemoriaLargoPlazo:
    def __init__(self, path="data/memoria_largo_plazo.json"):
        self.path = Path(path)
        self.items = []
        if self.path.exists():
            try:
                self.items = json.loads(self.path.read_text(encoding="utf-8"))
            except Exception:
                self.items = []

    def _persistir(self):
        self.path.parent.mkdir(parents=True, exist_ok=True)
        self.path.write_text(json.dumps(self.items, ensure_ascii=False), encoding="utf-8")

    def guardar(self, apoderado, texto, tipo="preferencia"):
        """Guarda un hecho con su embedding. Evita duplicados exactos. Devuelve (id, creado)."""
        for it in self.items:
            if it["apoderado"] == apoderado and it["texto"].strip().lower() == texto.strip().lower():
                return it["id"], False
        item = {"id": len(self.items) + 1, "apoderado": apoderado, "tipo": tipo,
                "texto": texto, "fecha": _ahora(), "vector": embed_textos([texto])[0].tolist()}
        self.items.append(item)
        self._persistir()
        return item["id"], True

    def recuperar(self, consulta, apoderado=None, top_k=3, umbral=None):
        """Recuperación semántica: devuelve los recuerdos más parecidos a la consulta."""
        umbral = UMBRAL_MEMORIA if umbral is None else umbral
        cand = [it for it in self.items if apoderado is None or it["apoderado"] == apoderado]
        if not cand:
            return []
        V = np.array([c["vector"] for c in cand])
        vq = embed_textos([consulta])[0]
        sims = V @ vq / (np.linalg.norm(V, axis=1) * np.linalg.norm(vq) + 1e-8)
        orden = np.argsort(-sims)[:top_k]
        return [{"texto": cand[i]["texto"], "tipo": cand[i]["tipo"], "fecha": cand[i]["fecha"],
                 "similitud": round(float(sims[i]), 3)} for i in orden if sims[i] >= umbral]

    def reiniciar(self):
        self.items = []
        if self.path.exists():
            self.path.unlink()

print("✅ Memoria de corto y largo plazo definida")


✅ Memoria de corto y largo plazo definida


In [11]:
# ===== 2.3 Planificación, decisiones y herramientas del agente =====

# ---------- Lógica de decisión (determinista, reutilizada por herramientas y pruebas) ----------
def decidir_retiro(isbn, sucursal_preferida=""):
    """Decide qué hacer con un ISBN según el stock y la sucursal preferida del apoderado."""
    disp = disponibilidad_por_isbn(CTX["stock"], isbn)
    if not disp:
        return {"accion": "sin_stock",
                "detalle": "Sin stock en ninguna sucursal: buscar alternativas y, si no hay, registrar encargo."}
    pref = _norm(sucursal_preferida)
    if pref:
        en_pref = [r for r in disp if pref in _norm(r["sucursal"])]
        if en_pref:
            r = max(en_pref, key=lambda x: x["cantidad"])
            return {"accion": "retirar_en_preferida", "sucursal": r["sucursal"], "cantidad": r["cantidad"],
                    "detalle": "Hay stock en la sucursal preferida."}
    mejor = max(disp, key=lambda x: x["cantidad"])
    detalle = "Retirar pronto: queda poco stock." if mejor["cantidad"] <= STOCK_CRITICO else "Stock suficiente."
    if pref:
        return {"accion": "retirar_en_otra_sucursal", "sucursal": mejor["sucursal"], "cantidad": mejor["cantidad"],
                "detalle": f"No hay stock en la sucursal preferida ({sucursal_preferida}). {detalle}"}
    return {"accion": "retirar_pronto" if mejor["cantidad"] <= STOCK_CRITICO else "retirar",
            "sucursal": mejor["sucursal"], "cantidad": mejor["cantidad"], "detalle": detalle}

def _nivel(producto):
    return producto["edad_curso"].split("/")[-1].strip()

def alternativas_con_stock(isbn, max_resultados=3):
    base = next((p for p in CTX["catalogo"] if p["isbn"] == isbn), None)
    if base is None:
        return []
    candidatos = []
    for p in CTX["catalogo"]:
        if p["isbn"] == isbn:
            continue
        disp = disponibilidad_por_isbn(CTX["stock"], p["isbn"])
        if not disp:
            continue
        puntaje = (2 if p["categoria"] == base["categoria"] else 0) + (1 if _nivel(p) == _nivel(base) else 0)
        if puntaje == 0:
            continue
        mejor = max(disp, key=lambda r: r["cantidad"])
        candidatos.append({"isbn": p["isbn"], "titulo": p["titulo"], "autor": p["autor"],
                           "categoria": p["categoria"], "curso": p["edad_curso"],
                           "sucursal": mejor["sucursal"], "cantidad": mejor["cantidad"], "afinidad": puntaje})
    candidatos.sort(key=lambda c: (-c["afinidad"], -c["cantidad"]))
    return candidatos[:max_resultados]

def planificar_tareas(lista, sucursal_preferida=""):
    """PLANIFICADOR: pre-evalúa cada título y ordena el trabajo del agente por prioridad.
       1 = stock crítico (retirar/reservar antes de que se agote)
       2 = disponible con stock normal
       3 = sin stock (requiere cadena: alternativas -> razonar reemplazo -> encargo)
       4 = no está en el catálogo (aclarar/derivar a ejecutivo, no inventar)"""
    plan = []
    for item in lista:
        top = buscar_con_score(CTX["indice"], item["titulo"], top_k=1)[0]
        base = {"titulo_lista": item["titulo"], "curso": item.get("curso", ""),
                "coseno": top["coseno"], "lexico": top["lexico"]}
        if not top["coincide"]:
            plan.append({**base, "prioridad": 4, "situacion": "no_encontrado", "isbn": None,
                         "pasos": ["buscar_en_catalogo", "registrar_resultado(no_encontrado)", "derivar a ejecutivo"]})
            continue
        prod = top["producto"]
        decision = decidir_retiro(prod["isbn"], sucursal_preferida)
        if decision["accion"] == "sin_stock":
            plan.append({**base, "titulo_catalogo": prod["titulo"], "isbn": prod["isbn"], "prioridad": 3,
                         "situacion": "sin_stock",
                         "pasos": ["consultar_stock", "buscar_alternativas", "razonar_reemplazo",
                                   "registrar_encargo (si no hay alternativas)", "registrar_resultado"]})
        else:
            critico = decision["cantidad"] <= STOCK_CRITICO
            plan.append({**base, "titulo_catalogo": prod["titulo"], "isbn": prod["isbn"],
                         "prioridad": 1 if critico else 2,
                         "situacion": "stock_critico" if critico else "disponible",
                         "sugerencia": decision,
                         "pasos": ["consultar_stock", "registrar_resultado"]})
    plan.sort(key=lambda p: (p["prioridad"], p["titulo_lista"]))
    for n, p in enumerate(plan, 1):
        p["orden"] = n
    return plan

# ---------- Herramientas de CONSULTA ----------
@tool
def buscar_en_catalogo(consulta: str) -> str:
    """Búsqueda semántica (RAG) en el catálogo de Librería Antártica. Recibe un título, aunque esté mal
    escrito. Devuelve los 3 candidatos más parecidos con isbn, autor, categoría, curso y el campo
    'coincide'. Si el primer candidato tiene coincide=false, el título NO está en el catálogo."""
    res = buscar_con_score(CTX["indice"], consulta, top_k=3)
    return _json([{"isbn": r["producto"]["isbn"], "titulo": r["producto"]["titulo"],
                   "autor": r["producto"]["autor"], "categoria": r["producto"]["categoria"],
                   "curso": r["producto"]["edad_curso"], "coincide": r["coincide"],
                   "coseno": r["coseno"], "lexico": r["lexico"]} for r in res])

@tool
def consultar_stock(isbn: str, sucursal_preferida: str = "") -> str:
    """Consulta el stock EXACTO por ISBN en todas las sucursales y devuelve además una 'decision'
    (retirar, retirar_pronto, retirar_en_otra_sucursal, sin_stock). Pasa sucursal_preferida si el
    apoderado la indicó o está en la memoria."""
    return _json({"isbn": isbn, "stock_por_sucursal": CTX["stock"].get(isbn, []),
                  "decision": decidir_retiro(isbn, sucursal_preferida)})

@tool
def buscar_alternativas(isbn: str) -> str:
    """Busca hasta 3 títulos de reemplazo con stock > 0 (misma categoría o mismo nivel) para un ISBN
    sin stock. Si devuelve lista vacía, no hay reemplazo posible: corresponde registrar un encargo."""
    return _json({"isbn_original": isbn, "alternativas": alternativas_con_stock(isbn)})

@tool
def recordar_contexto(consulta: str) -> str:
    """Memoria de LARGO plazo: recupera por similitud semántica lo que se sabe del apoderado actual
    (sucursal preferida, hijos y cursos, encargos previos)."""
    return _json(CTX["memoria"].recuperar(consulta, apoderado=CTX["apoderado"]))

@tool
def consultar_progreso() -> str:
    """Memoria de CORTO plazo: muestra el plan de la sesión, qué títulos ya tienen resultado y cuáles
    siguen pendientes. Úsala al retomar una tarea larga o ante preguntas de seguimiento."""
    est = estado_sesion()
    hechos = est["resultados"]
    pendientes = [p["titulo_lista"] for p in est["plan"] if p["titulo_lista"] not in hechos]
    return _json({"plan": [{"orden": p["orden"], "titulo": p["titulo_lista"], "prioridad": p["prioridad"]}
                           for p in est["plan"]], "resultados": hechos, "pendientes": pendientes})

# ---------- Herramienta de RAZONAMIENTO ----------
@tool
def planificar_lista(items_json: str = "", ruta_csv: str = "", sucursal_preferida: str = "") -> str:
    """Planifica la verificación de una lista escolar: evalúa cada título y devuelve el plan ordenado por
    prioridad (1 stock crítico, 2 normal, 3 sin stock, 4 no encontrado) con los pasos a seguir.
    Entrega la lista como items_json (texto JSON: [{"titulo": "...", "curso": "..."}]) o como ruta_csv
    (por ejemplo data/lista_escolar_compleja.csv). Llámala siempre antes de verificar una lista."""
    if ruta_csv:
        lista = cargar_lista_escolar(ruta_csv)
    else:
        lista = json.loads(items_json)
    plan = planificar_tareas(lista, sucursal_preferida)
    est = estado_sesion()
    est["plan"] = plan
    return _json({"total_titulos": len(plan), "plan": plan})

@tool
def razonar_reemplazo(titulo_original: str, curso: str, alternativas_json: str) -> str:
    """Razona cuál de las alternativas disponibles es el mejor reemplazo para un título sin stock,
    considerando curso, categoría y cantidad. Recibe la salida de buscar_alternativas. Devuelve
    {isbn, titulo, justificacion}; nunca propone un título que no esté en las alternativas."""
    alts = json.loads(alternativas_json)
    if isinstance(alts, dict):
        alts = alts.get("alternativas", [])
    # Validación anti-alucinación: solo se aceptan alternativas que existen en el catálogo y tienen stock
    validas = []
    for a in alts:
        isbn = a.get("isbn") if isinstance(a, dict) else None
        prod = next((p for p in CTX["catalogo"] if p["isbn"] == isbn), None)
        disp = disponibilidad_por_isbn(CTX["stock"], isbn) if prod else []
        if prod and disp:
            mejor = max(disp, key=lambda r: r["cantidad"])
            validas.append({"isbn": isbn, "titulo": prod["titulo"], "autor": prod["autor"],
                            "categoria": prod["categoria"], "curso": prod["edad_curso"],
                            "sucursal": mejor["sucursal"], "cantidad": mejor["cantidad"]})
    alts = validas
    if not alts:
        return _json({"isbn": None, "titulo": None,
                      "justificacion": "Ninguna alternativa entregada existe en el catálogo con stock. "
                                       "No hay reemplazo: corresponde registrar un encargo."})
    elegido = alts[0]
    justificacion = "Heurística: mayor afinidad de categoría/nivel y stock."
    try:
        prompt = (f"El título '{titulo_original}' (curso {curso}) no tiene stock. Alternativas con stock:\n"
                  f"{_json(alts)}\nElige UNA, solo de esa lista, priorizando el mismo curso y género. "
                  'Responde únicamente JSON: {"isbn": "...", "titulo": "...", "justificacion": "una frase"}')
        txt = _texto(CTX["llm"].invoke(prompt).content)
        dato = json.loads(re.search(r"\{.*\}", txt, re.S).group(0))
        match = next((a for a in alts if a["isbn"] == dato.get("isbn")), None)
        if match:
            elegido, justificacion = match, dato.get("justificacion", justificacion)
    except Exception:
        pass  # si el LLM falla, se usa la heurística
    return _json({"isbn": elegido["isbn"], "titulo": elegido["titulo"], "justificacion": justificacion})

# ---------- Herramientas de ESCRITURA ----------
@tool
def registrar_encargo(titulo: str, isbn: str, curso: str, motivo: str) -> str:
    """Escribe un encargo en data/encargos.csv cuando un título no tiene stock ni alternativas.
    No duplica encargos del mismo apoderado y título."""
    if disponibilidad_por_isbn(CTX["stock"], isbn):
        return _json({"registrado": False, "detalle": "El título tiene stock: no corresponde un encargo."})
    if alternativas_con_stock(isbn):
        return _json({"registrado": False, "detalle": "Hay alternativas con stock: ofrécelas con "
                                                      "razonar_reemplazo antes de registrar un encargo."})
    ruta = Path("data/encargos.csv")
    ruta.parent.mkdir(exist_ok=True)
    existentes = []
    if ruta.exists():
        with open(ruta, newline="", encoding="utf-8") as f:
            existentes = list(csv.DictReader(f))
    if any(e["apoderado"] == CTX["apoderado"] and e["isbn"] == isbn and e["titulo"] == titulo for e in existentes):
        return _json({"registrado": False, "detalle": "El encargo ya existía."})
    nuevo = not ruta.exists()
    with open(ruta, "a", newline="", encoding="utf-8") as f:
        w = csv.DictWriter(f, fieldnames=["fecha", "apoderado", "titulo", "isbn", "curso", "motivo", "estado"])
        if nuevo:
            w.writeheader()
        w.writerow({"fecha": _ahora(), "apoderado": CTX["apoderado"], "titulo": titulo, "isbn": isbn,
                    "curso": curso, "motivo": motivo, "estado": "pendiente"})
    return _json({"registrado": True, "archivo": str(ruta)})

@tool
def registrar_resultado(titulo: str, estado: str, detalle: str = "") -> str:
    """Escribe en la memoria de corto plazo el resultado de un título. estado: disponible | reemplazo |
    encargo | no_encontrado. Permite retomar la tarea sin repetir trabajo."""
    if estado not in ("disponible", "reemplazo", "encargo", "no_encontrado"):
        return _json({"ok": False, "error": "estado debe ser disponible, reemplazo, encargo o no_encontrado"})
    item = next((p for p in estado_sesion()["plan"] if _norm(p["titulo_lista"]) == _norm(titulo)), None)
    if item:  # coherencia con lo que el planificador verificó en el stock real
        sit = item["situacion"]
        if estado in ("reemplazo", "encargo") and sit in ("disponible", "stock_critico"):
            return _json({"ok": False, "error": "El título tiene stock: su estado correcto es 'disponible'."})
        if estado == "disponible" and sit in ("sin_stock", "no_encontrado"):
            return _json({"ok": False, "error": f"El título está en situación '{sit}': no puede ser 'disponible'."})
        if estado == "no_encontrado" and sit != "no_encontrado":
            return _json({"ok": False, "error": "El título sí existe en el catálogo."})
    estado_sesion()["resultados"][titulo] = {"estado": estado, "detalle": detalle}
    return _json({"ok": True, "titulo": titulo, "estado": estado})

@tool
def guardar_reporte_apoderado(contenido: str) -> str:
    """Guarda en la carpeta reportes/ el resumen final para el apoderado (archivo .md con fecha)."""
    carpeta = Path("reportes")
    carpeta.mkdir(exist_ok=True)
    ruta = carpeta / f"reporte_{CTX['apoderado']}_{datetime.datetime.now():%Y%m%d_%H%M%S}.md"
    ruta.write_text(f"# Reporte para {CTX['apoderado']}\n\n{contenido}\n", encoding="utf-8")
    return _json({"guardado": True, "archivo": str(ruta)})

@tool
def guardar_en_memoria(texto: str, tipo: str = "preferencia") -> str:
    """Memoria de LARGO plazo: guarda un dato estable del apoderado (sucursal preferida, hijo y curso,
    encargos hechos) para recordarlo en sesiones futuras. Redáctalo como frase completa y autónoma."""
    id_, creado = CTX["memoria"].guardar(CTX["apoderado"], texto, tipo)
    return _json({"id": id_, "creado": creado})

HERRAMIENTAS = [buscar_en_catalogo, consultar_stock, buscar_alternativas, recordar_contexto,
                consultar_progreso, planificar_lista, razonar_reemplazo, registrar_encargo,
                registrar_resultado, guardar_reporte_apoderado, guardar_en_memoria]

print(f"✅ {len(HERRAMIENTAS)} herramientas definidas:", [h.name for h in HERRAMIENTAS])


✅ 11 herramientas definidas: ['buscar_en_catalogo', 'consultar_stock', 'buscar_alternativas', 'recordar_contexto', 'consultar_progreso', 'planificar_lista', 'razonar_reemplazo', 'registrar_encargo', 'registrar_resultado', 'guardar_reporte_apoderado', 'guardar_en_memoria']


In [12]:
# ===== 2.4 Orquestación: ensamblado del agente (LangChain + LangGraph) =====
AGENT_SYSTEM_PROMPT = SYSTEM_PROMPT + """

PROCEDIMIENTO DEL AGENTE (síguelo en orden):
1. Si el mensaje trae [MEMORIA DE LARGO PLAZO], úsala (por ejemplo la sucursal preferida) y no vuelvas a
   preguntar lo que ya se sabe. Si el apoderado declara un dato estable (sucursal preferida, hijo y curso),
   guárdalo con guardar_en_memoria.
2. Si recibes una lista (en el texto o en un CSV), llama primero a planificar_lista y trabaja los títulos
   en el orden del plan (prioridad 1 = más urgente). Pasa sucursal_preferida si la conoces.
3. Para cada título aplica esta decisión:
   a) buscar_en_catalogo. Si el primer candidato tiene coincide=false, el título NO está en el catálogo:
      no inventes nada, registrar_resultado(no_encontrado) y ofrece derivar a un ejecutivo.
   b) consultar_stock. Si hay stock, registrar_resultado(disponible) indicando sucursal y cantidad; si queda
      poco stock, advierte que conviene retirarlo pronto; si no hay en la sucursal preferida, dilo y propón
      la sucursal que sí tiene.
   c) Si no hay stock: buscar_alternativas. Si hay, usa razonar_reemplazo y ofrece el reemplazo
      (registrar_resultado(reemplazo)). Si no hay ninguna, registrar_encargo y registrar_resultado(encargo).
4. Si las condiciones cambian (nueva sucursal preferida, título agregado o quitado), llama de nuevo a
   planificar_lista solo con lo afectado y usa consultar_progreso para saber qué falta. No repitas trabajo.
5. Al terminar una lista, redacta el resumen (máximo 8 líneas) y guárdalo con guardar_reporte_apoderado.
   Guarda en memoria lo relevante para el futuro (por ejemplo títulos encargados).
6. Reglas de consistencia (el sistema las verifica y rechaza lo que las incumpla):
   - Un título con poco stock sigue siendo "disponible"; "reemplazo" y "encargo" son SOLO para títulos sin stock.
   - Nunca inventes ISBN ni alternativas: a razonar_reemplazo pásale exactamente lo que devolvió buscar_alternativas.
   - Si buscar_alternativas devuelve alternativas, ofrécelas antes de registrar un encargo.
7. En preguntas de seguimiento usa lo ya conversado y consultar_progreso antes de volver a consultar."""

def inicializar_contexto():
    """Carga catálogo, índice vectorial, stock y memoria de largo plazo en el contexto compartido."""
    CTX["catalogo"] = cargar_catalogo()
    CTX["indice"] = construir_indice(CTX["catalogo"])
    CTX["stock"] = cargar_stock()
    CTX["memoria"] = MemoriaLargoPlazo()
    print(f"✅ Contexto listo: {len(CTX['catalogo'])} productos indexados, "
          f"{len(CTX['memoria'].items)} recuerdos en memoria de largo plazo")

def construir_agente(llm=None):
    llm = llm or obtener_llm()
    CTX["llm"] = llm
    middleware = []
    try:
        # Memoria de corto plazo acotada: resume la conversación cuando se alarga demasiado
        from langchain.agents.middleware import SummarizationMiddleware
        middleware = [SummarizationMiddleware(model=llm, trigger=("messages", 40), keep=("messages", 15))]
    except Exception as e:
        print("ℹ️ Sin resumen automático de historial:", e)
    return create_agent(model=llm, tools=HERRAMIENTAS, system_prompt=AGENT_SYSTEM_PROMPT,
                        checkpointer=InMemorySaver(),   # memoria de corto plazo por thread_id
                        middleware=middleware)

MOSTRAR_TRAZA = False   # True => además muestra el detalle de cada herramienta que usó el agente

def _mostrar_turno(mensaje, respuesta, recuerdos, traza, herramientas, segundos):
    """Salida limpia: pregunta del apoderado, respuesta del asistente y una línea de resumen."""
    from collections import Counter
    usadas = ", ".join(f"{n} ×{c}" for n, c in Counter(herramientas).items()) or "ninguna"
    texto = f"**👤 Apoderado:** {mensaje}\n\n"
    if recuerdos:
        texto += "🧠 *Memoria de largo plazo usada:* " + "; ".join(r["texto"] for r in recuerdos) + "\n\n"
    texto += f"**🤖 Asistente:**\n\n{respuesta}\n\n"
    texto += f"<sub>⏱️ {segundos:.1f} s · herramientas usadas: {usadas}</sub>"
    try:
        from IPython.display import display, Markdown
        display(Markdown(texto))
    except Exception:
        print(texto)
    if MOSTRAR_TRAZA:
        print("\n".join(traza))

def conversar(agente, mensaje, thread_id="sesion-1", apoderado="apoderado_demo", verbose=True):
    """Un turno de conversación. Mismo thread_id = misma sesión (memoria de corto plazo).
    Antes de invocar, recupera de la memoria de largo plazo lo relevante para el mensaje."""
    t0 = time.time()
    CTX["thread_id"], CTX["apoderado"] = thread_id, apoderado
    try:
        recuerdos = CTX["memoria"].recuperar(mensaje, apoderado=apoderado)
        if not recuerdos:   # respaldo: si ningún recuerdo supera el umbral, entrega los 2 más cercanos
            recuerdos = CTX["memoria"].recuperar(mensaje, apoderado=apoderado, top_k=2, umbral=-1)
    except Exception as e:
        recuerdos = []
        print("ℹ️ No se pudo consultar la memoria de largo plazo:", e)
    texto = mensaje
    if recuerdos:
        bloque = "\n".join(f"- {r['texto']} (guardado {r['fecha']}, similitud {r['similitud']})" for r in recuerdos)
        texto = f"[MEMORIA DE LARGO PLAZO]\n{bloque}\n\n[MENSAJE DEL APODERADO]\n{mensaje}"
    config = {"configurable": {"thread_id": thread_id}, "recursion_limit": 60}
    previos = len(agente.get_state(config).values.get("messages", []))
    salida = agente.invoke({"messages": [{"role": "user", "content": texto}]}, config=config)
    nuevos = salida["messages"][previos:]

    herramientas, traza = [], []
    for m in nuevos:
        if getattr(m, "tool_calls", None):
            for tc in m.tool_calls:
                herramientas.append(tc["name"])
                traza.append(f"🔧 {tc['name']}({_json(tc['args'])[:160]})")
        elif m.type == "tool":
            traza.append(f"   ↳ {_texto(m.content)[:200]}")
    respuesta = _texto(nuevos[-1].content) if nuevos else ""
    if verbose:
        _mostrar_turno(mensaje, respuesta, recuerdos, traza, herramientas, time.time() - t0)
    return {"respuesta": respuesta, "herramientas": herramientas, "traza": traza, "recuerdos": recuerdos}

# Inicialización (hace llamadas a la API de Gemini: embeddings del catálogo y prueba del modelo)
inicializar_contexto()
try:
    agente = construir_agente()
except Exception as e:
    agente = None
    print(f"⚠️ No se pudo crear el agente: {e}")


✅ Contexto listo: 10 productos indexados, 1 recuerdos en memoria de largo plazo
✅ Modelo LLM en uso (Gemini): gemini-3.8-flash


In [13]:
# ===== 2.5 Pruebas de software de las herramientas y la planificación (sin LLM) =====
os.makedirs("evidencias", exist_ok=True)
resultados_pruebas = []

def prueba(nombre, condicion, detalle=""):
    resultados_pruebas.append({"prueba": nombre, "ok": bool(condicion), "detalle": detalle})
    print(("✅" if condicion else "❌"), nombre, "-", detalle)

# Reinicia los archivos que escribe el agente para que las pruebas sean reproducibles
for ruta in ["data/encargos.csv"]:
    Path(ruta).unlink(missing_ok=True)
CTX["memoria"].reiniciar()
ESTADOS.clear()

# 1. Búsqueda semántica con error de escritura
r = buscar_con_score(CTX["indice"], "papelucho 3 basico", top_k=1)[0]
prueba("Búsqueda tolera variaciones", r["producto"]["titulo"] == "Papelucho" and r["coincide"],
       f"coseno={r['coseno']} léxico={r['lexico']}")

# 2. Título inexistente: no se inventa
r = buscar_con_score(CTX["indice"], "Harry Potter y la piedra filosofal", top_k=1)[0]
prueba("Título inexistente se detecta", not r["coincide"],
       f"mejor candidato='{r['producto']['titulo']}' coseno={r['coseno']} léxico={r['lexico']} (si falla, ajusta UMBRAL_COSENO)")

# 3. Consulta exacta de stock
d = decidir_retiro("9789561234567")
prueba("Stock Papelucho = 5 en Mall Costanera", d["sucursal"] == "Mall Costanera Center" and d["cantidad"] == 5, str(d))
d = decidir_retiro("9789561234569")
prueba("Sub Terra sin stock", d["accion"] == "sin_stock", str(d))

# 4. Decisión adaptativa según sucursal preferida
d = decidir_retiro("9789561234567", "Plaza Vespucio")
prueba("Sin stock en la preferida => propone otra sucursal", d["accion"] == "retirar_en_otra_sucursal", str(d))
d = decidir_retiro("9789561234570", "Plaza Vespucio")
prueba("Con stock en la preferida => retira ahí", d["accion"] == "retirar_en_preferida", str(d))
d = decidir_retiro("9789561234568")
prueba("Stock bajo => retirar pronto", d["accion"] == "retirar_pronto", str(d))

# 5. Alternativas
alts = alternativas_con_stock("9789561234569")
prueba("Alternativas para Sub Terra con stock", len(alts) > 0 and all(a["cantidad"] > 0 for a in alts),
       str([a["titulo"] for a in alts]))
prueba("Diario de Ana Frank no tiene alternativas", alternativas_con_stock("9789561234574") == [])

# 6. Planificador: orden por prioridad
plan = planificar_tareas(cargar_lista_escolar("data/lista_escolar_compleja.csv"), "Plaza Vespucio")
orden = [(p["titulo_lista"], p["prioridad"]) for p in plan]
print("   Plan:", orden)
prioridades = [p["prioridad"] for p in plan]
prueba("Plan ordenado por prioridad", prioridades == sorted(prioridades), str(prioridades))
prueba("El título inexistente queda al final (prioridad 4)", plan[-1]["situacion"] == "no_encontrado", plan[-1]["titulo_lista"])
prueba("Stock crítico va primero", plan[0]["situacion"] == "stock_critico", plan[0]["titulo_lista"])

# 6b. Guardas anti-alucinación del agente
estado_sesion("prueba_guardas")["plan"] = plan
CTX["thread_id"] = "prueba_guardas"
r = json.loads(registrar_resultado.invoke({"titulo": "El Principito", "estado": "reemplazo"}))
prueba("No acepta 'reemplazo' si el título tiene stock", not r["ok"], r.get("error", ""))
r = json.loads(razonar_reemplazo.invoke({"titulo_original": "X", "curso": "",
        "alternativas_json": '[{"isbn": "9789561234582", "titulo": "Inventado"}]'}))
prueba("Rechaza alternativas inventadas", r["isbn"] is None, r["justificacion"][:60])
r = json.loads(registrar_encargo.invoke({"titulo": "Sub Terra", "isbn": "9789561234569", "curso": "1° Medio", "motivo": "x"}))
prueba("No registra encargo si hay alternativas con stock", not r["registrado"], r["detalle"][:60])
CTX["thread_id"] = "default"

# 7. Memoria de largo plazo (escritura + recuperación semántica + persistencia)
CTX["memoria"].guardar("apoderado_test", "Prefiere retirar los libros en la sucursal Plaza Vespucio", "preferencia")
CTX["memoria"].guardar("apoderado_test", "Tiene una hija en 4° Básico y un hijo en 8° Básico", "familia")
rec = CTX["memoria"].recuperar("¿en qué sucursal quiere retirar?", apoderado="apoderado_test", top_k=1, umbral=-1)  # prueba el ranking, independiente del umbral
prueba("Memoria LP recupera la preferencia por significado", bool(rec) and "Plaza Vespucio" in rec[0]["texto"], str(rec))
prueba("Memoria LP persiste en disco", len(MemoriaLargoPlazo().items) == 2)
prueba("Memoria LP no mezcla apoderados", CTX["memoria"].recuperar("sucursal", apoderado="otro") == [])
CTX["memoria"].reiniciar()

# 8. Escritura idempotente de encargos
CTX["apoderado"] = "apoderado_test"
a = json.loads(registrar_encargo.invoke({"titulo": "X", "isbn": "1", "curso": "1° Medio", "motivo": "prueba"}))
b = json.loads(registrar_encargo.invoke({"titulo": "X", "isbn": "1", "curso": "1° Medio", "motivo": "prueba"}))
prueba("Encargo se registra una sola vez", a["registrado"] and not b["registrado"])
Path("data/encargos.csv").unlink(missing_ok=True)

ok = sum(p["ok"] for p in resultados_pruebas)
print(f"\nResumen: {ok}/{len(resultados_pruebas)} pruebas aprobadas")
with open("evidencias/pruebas_unitarias.md", "w", encoding="utf-8") as f:
    f.write(f"# Pruebas unitarias ({_ahora()})\n\n| Prueba | Resultado | Detalle |\n|---|---|---|\n")
    for p in resultados_pruebas:
        f.write(f"| {p['prueba']} | {'✅' if p['ok'] else '❌'} | {p['detalle']} |\n")
print("📝 Evidencia guardada en evidencias/pruebas_unitarias.md")


✅ Búsqueda tolera variaciones - coseno=0.705 léxico=1.0
✅ Título inexistente se detecta - mejor candidato='Matilda' coseno=0.64 léxico=0.278 (si falla, ajusta UMBRAL_COSENO)
✅ Stock Papelucho = 5 en Mall Costanera - {'accion': 'retirar', 'sucursal': 'Mall Costanera Center', 'cantidad': 5, 'detalle': 'Stock suficiente.'}
✅ Sub Terra sin stock - {'accion': 'sin_stock', 'detalle': 'Sin stock en ninguna sucursal: buscar alternativas y, si no hay, registrar encargo.'}
✅ Sin stock en la preferida => propone otra sucursal - {'accion': 'retirar_en_otra_sucursal', 'sucursal': 'Mall Costanera Center', 'cantidad': 5, 'detalle': 'No hay stock en la sucursal preferida (Plaza Vespucio). Stock suficiente.'}
✅ Con stock en la preferida => retira ahí - {'accion': 'retirar_en_preferida', 'sucursal': 'Plaza Vespucio', 'cantidad': 6, 'detalle': 'Hay stock en la sucursal preferida.'}
✅ Stock bajo => retirar pronto - {'accion': 'retirar_pronto', 'sucursal': 'Mall Costanera Center', 'cantidad': 2, 'detalle':

In [15]:
# ===== 2.6 Demostración y pruebas del agente (escenarios con LLM) =====
# Escenarios: (1) lista completa con planificación, (2) seguimiento con memoria de corto plazo,
# (3) cambio de condiciones -> replanificación, (4) NUEVA sesión con memoria de largo plazo,
# (5) título inexistente sin inventar datos.

# Elige qué escenarios ejecutar para no esperar los 5 (el 2, 3 y 4 dependen de que antes se haya corrido el 1)
ESCENARIOS_A_EJECUTAR = [1]

def limpiar_estado_demo():
    """Deja el entorno limpio para que la demostración sea reproducible."""
    Path("data/encargos.csv").unlink(missing_ok=True)
    CTX["memoria"].reiniciar()
    ESTADOS.clear()

evidencia_agente = []

def escenario(nombre, mensaje, thread, esperado_herramientas=(), no_debe_contener=()):
    if int(nombre.split(".")[0]) not in ESCENARIOS_A_EJECUTAR:
        print(f"⏭️ Escenario omitido: {nombre}")
        return None
    print(f"\n{'=' * 80}\n🎬 {nombre}\n{'=' * 80}")
    r = conversar(agente, mensaje, thread_id=thread, apoderado="apoderado_demo")
    faltan = [h for h in esperado_herramientas if h not in r["herramientas"]]
    prohibidos = [t for t in no_debe_contener if t.lower() in r["respuesta"].lower()]
    ok = not faltan and not prohibidos
    print(("✅" if ok else "❌"), "Verificación:",
          f"herramientas esperadas ausentes={faltan}" if faltan else "usó las herramientas esperadas",
          f"| texto prohibido presente={prohibidos}" if prohibidos else "")
    evidencia_agente.append({"escenario": nombre, "mensaje": mensaje, "ok": ok, **r})
    return r

if agente is None:
    print("⚠️ El agente no está disponible: revisa la API key / modelo en la celda anterior.")
else:
    if 1 in ESCENARIOS_A_EJECUTAR:
        limpiar_estado_demo()

    escenario("1. Lista completa: planificación + decisiones por título",
              "Hola, soy apoderado de Valentina y Tomás. Prefiero retirar en Plaza Vespucio. "
              "Verifica la lista que está en data/lista_escolar_compleja.csv y arma el resumen.",
              "sesion-1",
              esperado_herramientas=["planificar_lista", "buscar_en_catalogo", "consultar_stock",
                                     "guardar_reporte_apoderado", "guardar_en_memoria"])

    escenario("2. Seguimiento (memoria de corto plazo, mismo hilo)",
              "De todos esos libros, ¿cuál tenía poco stock y cuál quedó como encargo?",
              "sesion-1")

    escenario("3. Cambio de condiciones -> replanificación",
              "Cambié de idea: ahora prefiero retirar todo en Mall Costanera Center. ¿Cambia algo?",
              "sesion-1",
              esperado_herramientas=["consultar_stock"])

    escenario("4. Nueva sesión: memoria de largo plazo",
              "Hola, soy yo de nuevo. ¿Qué sucursal prefería y qué libros dejé encargados?",
              "sesion-2")

    escenario("5. Título inexistente: no inventa datos",
              "¿Tienen 'Cien años de soledad' de Gabriel García Márquez?",
              "sesion-2",
              esperado_herramientas=["buscar_en_catalogo"])

    aprobados = sum(e["ok"] for e in evidencia_agente)
    print(f"\nResumen: {aprobados}/{len(evidencia_agente)} escenarios cumplieron las verificaciones automáticas")
    print("Revisa además la traza (🔧) para evidenciar las decisiones del agente en cada escenario.")

    with open("evidencias/escenarios_agente.md", "w", encoding="utf-8") as f:
        f.write(f"# Escenarios del agente ({_ahora()})\n")
        for e in evidencia_agente:
            f.write(f"\n## {e['escenario']} {'✅' if e['ok'] else '❌'}\n\n**Mensaje:** {e['mensaje']}\n\n")
            f.write("**Traza de herramientas:**\n```\n" + "\n".join(e["traza"]) + "\n```\n\n")
            f.write(f"**Respuesta del agente:**\n\n{e['respuesta']}\n")
    print("📝 Evidencia guardada en evidencias/escenarios_agente.md")



🎬 1. Lista completa: planificación + decisiones por título


GoogleRateLimitError: Error calling model 'gemini-3.8-flash' (RESOURCE_EXHAUSTED): 429 RESOURCE_EXHAUSTED. {'error': {'code': 429, 'message': 'You exceeded your current quota, please check your plan and billing details. For more information on this error, head to: https://aistudio.google.com/docs/rate-limits. To monitor your current usage, head to: https://ai.dev/rate-limit. \n* Quota exceeded for metric: generativelanguage.googleapis.com/generate_content_free_tier_requests, limit: 20, model: gemini-3.8-flash, description: Generate content requests per day limit exceeded on Free Tier. See https://aistudio.google.com/docs/rate-limits.\nPlease retry in 6h47m51.195936661s.', 'status': 'RESOURCE_EXHAUSTED', 'details': [{'@type': 'type.googleapis.com/google.rpc.Help', 'links': [{'description': 'Learn more about Gemini API quotas', 'url': 'https://aistudio.google.com/docs/rate-limits'}]}, {'@type': 'type.googleapis.com/google.rpc.QuotaFailure', 'violations': [{'description': 'Generate conten

---
## 2.7 Orquestación y justificación de componentes

```
Apoderado ──► conversar() ──► [recupera memoria de largo plazo (semántica)]
                    │
                    ▼
        Agente ReAct (LangChain create_agent / LangGraph)  ◄── checkpointer: memoria de corto plazo (thread_id)
                    │  decide qué herramienta usar en cada paso
     ┌──────────────┼───────────────────────────┬──────────────────────┐
     ▼              ▼                           ▼                      ▼
 CONSULTA       RAZONAMIENTO                 ESCRITURA              MEMORIA
 buscar_en_     planificar_lista             registrar_encargo      guardar_en_memoria
  catalogo      razonar_reemplazo            registrar_resultado    recordar_contexto
 consultar_                                  guardar_reporte_       consultar_progreso
  stock                                       apoderado
 buscar_alternativas
```

| Componente | Decisión de diseño | Justificación respecto del flujo de trabajo |
|---|---|---|
| **LangChain + LangGraph** (`create_agent`) | Framework de agentes con bucle ReAct y checkpointer | Provee tool-calling, estado por hilo y persistencia sin reinventar el bucle; escala a más herramientas o a multi-agente y es compatible con Gemini vía `langchain-google-genai`. |
| **Groq / Mistral / Gemini** (LLM) + embeddings | Proveedor intercambiable con `PROVEEDOR`; con Groq los embeddings son locales (fastembed) | Los tres ofrecen *tool-calling*; Groq tiene nivel gratuito sin tarjeta y los embeddings locales no dependen de ninguna API. |
| **RAG sobre el catálogo** | Búsqueda semántica + coincidencia textual | Las listas escolares traen errores de escritura; el umbral (`coincide`) evita aceptar un título que no existe. |
| **Stock por consulta exacta** | Herramienta determinista (no el LLM) | La disponibilidad no puede inventarse: es un dato transaccional. |
| **Planificador determinista** | Ordena por prioridad (crítico → normal → sin stock → no encontrado) | Atiende primero lo que se puede agotar y deja para el final lo que requiere más pasos. |
| **Memoria de corto plazo** | Checkpointer + resumen automático + `EstadoSesion` | Mantiene coherencia en listas largas y permite retomar sin repetir consultas. |
| **Memoria de largo plazo** | JSON + embeddings + similitud coseno | Recuerda preferencias entre sesiones; es simple, auditable y no requiere servidor. Para producción se reemplazaría por una base vectorial. |
| **Herramientas de escritura** | Archivos CSV/MD | Simulan la integración con sistemas internos (encargos y reportes). |

### Limitaciones de la Parte 2
- Los datos siguen siendo simulados y el índice vectorial vive en memoria.
- Los umbrales `UMBRAL_COSENO` y `UMBRAL_LEXICO` deben calibrarse con el catálogo real.
- El checkpointer `InMemorySaver` se pierde al reiniciar el kernel (la memoria de largo plazo sí persiste en `data/`).
